In [2]:
!pip install kumoai

In [2]:
!gcloud auth application-default login

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=kQJMPLfavON8jZgRkhx7zbETT1qcY8&prompt=consent&token_usage=remote&access_type=offline&code_challenge=d-q6yeUTt6ePkrbY5Vf5YuEWOOwgiJFl4G1ovy8aNP0&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AdkVLPzzvkBik3Rc9U-eF6c3-KrOtNxzLpnLfDJKo7Mj8TgHdJd7iGQVNSXNOp3HRCddag

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Ca

In [3]:
import os
import kumoai.experimental.rfm as rfm, os
import pandas as pd

from google.cloud import bigquery

In [5]:
KUMO_API_KEY = os.environ.get("KUMO_API_KEY")
rfm.init(api_key=KUMO_API_KEY)

INFO:kumoai:Initialized Kumo SDK v2.22.0 against deployment 'https://kumorfm.ai/api'


In [6]:
client = bigquery.Client(project="diesel-charge-465712-f8")

# -----------------------------
# 1. Build a COHORT (anchor)
# -----------------------------
cohort_query = """
WITH cohort AS (
  SELECT
    subject_id,
    hadm_id
  FROM `physionet-data.mimiciv_3_1_hosp.admissions`
  WHERE RAND() < 0.01   -- ~1% sample (adjust if needed)
)

SELECT * FROM cohort
"""

cohort_df = client.query(cohort_query).to_dataframe()

hadm_ids = cohort_df["hadm_id"].tolist()
subject_ids = cohort_df["subject_id"].tolist()


# -----------------------------
# 2. Admissions (label source)
# -----------------------------
admissions_query = """
SELECT *
FROM `physionet-data.mimiciv_3_1_hosp.admissions`
WHERE hadm_id IN UNNEST(@hadm_ids)
"""

admissions_job = client.query(
    admissions_query,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter("hadm_ids", "INT64", hadm_ids)
        ]
    )
)

admissions_df = admissions_job.to_dataframe()


# -----------------------------
# 3. Patients (demographics)
# -----------------------------
patients_query = """
SELECT *
FROM `physionet-data.mimiciv_3_1_hosp.patients`
WHERE subject_id IN UNNEST(@subject_ids)
"""

patients_job = client.query(
    patients_query,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter("subject_ids", "INT64", subject_ids)
        ]
    )
)

patients_df = patients_job.to_dataframe()


# -----------------------------
# 4. ICU stays (filtered)
# -----------------------------
icustays_query = """
SELECT *
FROM `physionet-data.mimiciv_3_1_icu.icustays`
WHERE hadm_id IN UNNEST(@hadm_ids)
"""

icustays_job = client.query(
    icustays_query,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter("hadm_ids", "INT64", hadm_ids)
        ]
    )
)

icustays_df = icustays_job.to_dataframe()


# -----------------------------
# 5. Labevents (SAFE FILTERED)
# -----------------------------
labevents_query = """
SELECT l.*
FROM `physionet-data.mimiciv_3_1_hosp.labevents` l
WHERE l.subject_id IN UNNEST(@subject_ids)
"""

labevents_job = client.query(
    labevents_query,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter("subject_ids", "INT64", subject_ids)
        ]
    )
)

labevents_df = labevents_job.to_dataframe()

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [32]:
tables_dict = {
    'admissions': admissions_df,
    'patients': patients_df,
    'icustays': icustays_df,
    'labevents': labevents_df
}

In [8]:
def preprocess_tables_dict(tables_dict):
    for table_name, df in tables_dict.items():
      dbdate_columns = df.dtypes[df.dtypes == 'dbdate'].index.tolist()
      for col in dbdate_columns:
        df[col] = pd.to_datetime(df[col])

    return tables_dict

In [33]:
icustays_df = icustays_df.merge(
    admissions_df[['hadm_id', 'deathtime']],
    on='hadm_id',
    how='left'
)

# 2. Convert to datetime objects for calculation
icustays_df['intime'] = pd.to_datetime(icustays_df['intime'])
icustays_df['deathtime'] = pd.to_datetime(icustays_df['deathtime'])

# 3. Calculate if they died within 48 hours of ICU admission
icustays_df['died_within_48h'] = (
    icustays_df['deathtime'].notna() &
    ((icustays_df['deathtime'] - icustays_df['intime']).dt.total_seconds() / 3600.0 <= 48.0)
).astype(int) # 1 for True, 0 for False

# Update your tables dictionary with the modified dataframe
tables_dict['icustays'] = icustays_df

In [34]:
tables_dict = preprocess_tables_dict(tables_dict)

In [35]:
def transform_to_rfm_table(tables_dict):
  result = {}
  for table_name, df in tables_dict.items():
    rfm_table = rfm.LocalTable(
      df = df,
      name = table_name
    ).infer_metadata()
    result[table_name] = rfm_table
  return result

In [37]:
# tables = transform_to_rfm_table(tables_dict)
# tables['admissions'].primary_key = "hadm_id"
# tables['icustays'].primary_key = "stay_id"
# tables['patients'].primary_key = "subject_id"
# tables['labevents'].primary_key = "labevent_id"
# graph = rfm.LocalGraph(tables=tables.values())
# graph.link(
#     src_table="admissions",
#     fkey="subject_id",
#     dst_table="patients"
# )

# graph.link(
#     src_table="icustays",
#     fkey="hadm_id",
#     dst_table="admissions"
# )

# graph.link(
#     src_table="labevents",
#     fkey="subject_id",
#     dst_table="patients"
# )

# graph.visualize()


tables = transform_to_rfm_table(tables_dict)

tables['patients'].primary_key = "subject_id"
tables['admissions'].primary_key = "hadm_id"
tables['icustays'].primary_key = "stay_id"
tables['labevents'].primary_key = "labevent_id"

graph = rfm.LocalGraph(tables=tables.values())

# Link child tables to parent tables
graph.link(src_table="admissions", fkey="subject_id", dst_table="patients")
graph.link(src_table="icustays", fkey="hadm_id", dst_table="admissions")
graph.link(src_table="labevents", fkey="hadm_id", dst_table="admissions")

model = rfm.KumoRFM(graph=graph)

Inferred time column `edregtime` for table `admissions`

Inferred time column `dod` for table `patients`

Inferred time column `intime` for table `icustays`

Inferred primary key `labevent_id` and time column `charttime` for table `labevents`

Output()

In [22]:
model = rfm.KumoRFM(graph)

Output()

In [40]:
# The query is now clean, simple, and perfectly valid PQL syntax
query = """
PREDICT icustays.died_within_48h
FOR EACH icustays.stay_id
"""

patient_ids = tables_dict['icustays']['stay_id'].dropna().astype(int).tolist()

with model.batch_mode(batch_size="max", num_retries=1):
    try:
        df = model.predict(query, indices=patient_ids)
        display(df.head())
    except Exception as e:
        print(f"Error: {e}")

Output()

,ENTITY,ANCHOR_TIMESTAMP,CLASS,SCORE,PREDICTED
0,30300387,2214-05-12 06:30:00,0,0.999064,True
1,30300387,2214-05-12 06:30:00,1,0.000936,False
2,39375574,2214-05-12 06:30:00,0,0.999452,True
3,39375574,2214-05-12 06:30:00,1,0.000548,False
4,37132939,2214-05-12 06:30:00,0,0.999711,True


In [48]:
metrics = model.evaluate(
    query,
    metrics=['acc', 'precision', 'recall', 'f1', 'mrr']
)

metrics

Output()

,metric,value
0,acc,0.979123
1,f1,0.716903
2,mrr,0.989562
3,precision,0.716903
4,recall,0.716903
